In [ ]:
import torch
import torch.optim as optim
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import confusion_matrix
import seaborn as sns
import pandas as pd
import csv

import dataset
import datasplit
import model.models as models
import trainer
import utils

pd.set_option("display.max_rows", None, "display.max_columns", None)

In [ ]:
models_folder = '../../models_and_results/models'
model_name = '20201027_fxnet_mono_disc_best'
results_folder = '../../models_and_results/results'
results_subfolder = '20201027_fxnet_mono_disc'

all_train_losses = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'all_train_losses.npy')), allow_pickle=True)
all_train_correct = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'all_train_correct.npy')), allow_pickle=True)
best_train_results = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'best_train_results.npy')), allow_pickle=True)

all_val_losses = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'all_val_losses.npy')), allow_pickle=True)
all_val_correct = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'all_val_correct.npy')), allow_pickle=True)
best_val_results = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'best_val_results.npy')), allow_pickle=True)

all_test_losses = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'all_test_losses.npy')), allow_pickle=True)
all_test_correct = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'all_test_correct.npy')), allow_pickle=True)
best_test_results = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'best_test_results.npy')), allow_pickle=True)

fx_labels = np.load(file=('%s/%s/%s' % (results_folder, results_subfolder, 'fx_labels.npy')), allow_pickle=True)

train_set_size = best_train_results.shape[0]
val_set_size = best_val_results.shape[0]
test_set_size = best_test_results.shape[0]

In [ ]:
best_test_results[0]

In [ ]:
fx_labels

In [ ]:
# CONFUSION MATRICES

# Find best performance epoch based on validation
# Extract results from training and test for that epoch
best_val = 100 * max(all_val_correct) / val_set_size
best_val_idx = np.argmax(all_val_correct)

print("Best Validation Accuracy: ",best_val, "at epoch ", best_val_idx)
print("Training Accuracy: ", 100 * all_train_correct[best_val_idx] / train_set_size)
print("Test Accuracy: ", 100 * all_test_correct[best_val_idx] / test_set_size)

# Confusion matrices for best validation epoch
# extract predictions and targets vectors
train_preds = best_train_results[:, 2]
train_targets = best_train_results[:, 3]
val_preds = best_val_results[:, 2]
val_targets = best_val_results[:, 3]
test_preds = best_test_results[:, 2]
test_targets = best_test_results[:, 3]

# calculate conf. matrix
train_cm = confusion_matrix(y_true=train_targets, y_pred=train_preds, labels=['0','1','2','3','4','5','6','7','8','9','10','11','12'])
val_cm = confusion_matrix(y_true=val_targets, y_pred=val_preds, labels=['0','1','2','3','4','5','6','7','8','9','10','11','12'])
test_cm = confusion_matrix(y_true=test_targets, y_pred=test_preds, labels=['0','1','2','3','4','5','6','7','8','9','10','11','12'])

In [ ]:
print(test_cm)

In [ ]:
# Test CM
plt.figure(figsize=(10,10))

utils.plot_confusion_matrix(test_cm, fx_labels, normalize=False)

plt.xlabel('True label', fontsize=16)
plt.ylabel('Predicted label', fontsize=16)
# plt.title('FxNet - train = mono - test = mono', fontsize=20)
plt.rc('xtick', labelsize=16)
plt.rc('ytick', labelsize=16)
plt.rc('font', size=16)


# plt.savefig('%s/%s/%s' % (results_folder, results_subfolder, 'fxnet_test_cm_norm.png'))

In [ ]:
# extract vectors for dataframe
test_indexes = best_test_results[:, 0]
test_filenames = best_test_results[:, 1] 
test_preds = np.stack(best_test_results[:, 2])
test_targets = np.stack(best_test_results[:, 3])

test_df = pd.DataFrame(data={
                                'index': test_indexes, 
                                'filename': test_filenames,
                                'fx_pred': test_preds,
                                'fx_targ': test_targets 
                            })
test_df[:5]

In [ ]:
fx_labels

In [ ]:
# select one effect to analyse - change fx label string accordingly
fx1 = test_df[test_df['filename'].str.contains('-DPL-')]
fx2 = test_df[test_df['filename'].str.contains('-RAT-')]

In [ ]:
fx1.head()

In [ ]:
fx2.head()

In [ ]:
# select wrong predictions - change effect label number accordingly (e.g. 10 = SD1)
fx1_wrong_pred = fx1[fx1['fx_pred'].str.contains('8')]
fx2_wrong_pred = fx2[fx2['fx_pred'].str.contains('3')]

In [ ]:
# print number of wrong predictions as a function of parameters
print('D10: ',  len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D10')]))
print('D8: ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D8')]))
print('D5: ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D5')]))
print('D2: ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D2')]))
print()
# print('D10 T10: ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D10T10-')]))
# print('D10 T8 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D10T8-')]))
# print('D10 T5 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D10T5-')]))
# print('D10 T2 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D10T2-')]))
# print('D10 T0 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D10T0-')]))
# print()
# print('D8 T10: ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D8T10-')]))
# print('D8 T8 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D8T8-')]))
# print('D8 T5 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D8T5-')]))
# print('D8 T2 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D8T2-')]))
# print('D8 T0 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D8T0-')]))
# print()
# print('D5 T10: ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D5T10-')]))
# print('D5 T8 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D5T8-')]))
# print('D5 T5 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D5T5-')]))
# print('D5 T2 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D5T2-')]))
# print('D5 T0 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D5T0-')]))
# print()
# print('D2 T10: ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D2T10-')]))
# print('D2 T8 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D2T8-')]))
# print('D2 T5 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D2T5-')]))
# print('D2 T2 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D2T2-')]))
# print('D2 T0 : ',   len(fx1_wrong_pred[fx1_wrong_pred['filename'].str.contains('-D2T0-')]))

In [ ]:
# print number of wrong predictions as a function of parameters
print('D10: ',  len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D10')]))
print('D8: ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D8')]))
print('D5: ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D5')]))
print('D2: ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D2')]))
print()
print('D10 T10: ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D10F10-')]))
print('D10 T8 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D10F8-')]))
print('D10 T5 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D10F5-')]))
print('D10 T2 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D10F2-')]))
print('D10 T0 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D10F0-')]))
print()
print('D8 T10: ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D8F10-')]))
print('D8 T8 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D8F8-')]))
print('D8 T5 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D8F5-')]))
print('D8 T2 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D8F2-')]))
print('D8 T0 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D8F0-')]))
print()
print('D5 T10: ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D5F10-')]))
print('D5 T8 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D5F8-')]))
print('D5 T5 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D5F5-')]))
print('D5 T2 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D5F2-')]))
print('D5 T0 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D5F0-')]))
print()
print('D2 T10: ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D2F10-')]))
print('D2 T8 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D2F8-')]))
print('D2 T5 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D2F5-')]))
print('D2 T2 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D2F2-')]))
print('D2 T0 : ',   len(fx2_wrong_pred[fx2_wrong_pred['filename'].str.contains('-D2F0-')]))

In [ ]:
# print wrong predictions where D = 5
print(wrong_df[wrong_df['filename'].str.contains('-D10-')])

In [ ]:
# print wrong predictions where D = 5
print(wrong_df[wrong_df['filename'].str.contains('-D2-')])

In [ ]:
import shutil, os

in_folder_DPL = '/Volumes/STEVE/DATASETS/GUITAR-FX-DIST/DATA/Mono_Discrete/Audio/DPL'
in_folder_RAT = '/Volumes/STEVE/DATASETS/GUITAR-FX-DIST/DATA/Mono_Discrete/Audio/RAT'
out_folder_DPL = '/Users/Marco/Desktop/GFX_listening_test/DPL'
out_folder_RAT = '/Users/Marco/Desktop/GFX_listening_test/RAT'

wrong_df_sub = wrong_df[wrong_df['filename'].str.contains('-D10-')]
for filename in wrong_df_sub['filename']:
    shutil.copy('%s/%s%s' % (in_folder_DPL,filename,'.wav'), '%s/%s%s' % (out_folder_DPL,filename,'.wav'))

In [ ]:
print(filename[:9]+'-SD1-'+'D2T0-'+filename[-5:])

In [ ]:
filename = filename[14] = 'D'

In [ ]:
filename[17]

In [ ]:
wrong_df_sub = wrong_df[wrong_df['filename'].str.contains('-D10-')]
for filename in wrong_df_sub['filename']:
    filename = filename[:9]+'-RAT-'+'D2F0-'+filename[-5:]
    shutil.copy('%s/%s%s' % (in_folder_RAT,filename,'.wav'), '%s/%s%s' % (out_folder_RAT,filename,'.wav'))
    filename = filename[:9]+'-RAT-'+'D5F0-'+filename[-5:]
    shutil.copy('%s/%s%s' % (in_folder_RAT,filename,'.wav'), '%s/%s%s' % (out_folder_RAT,filename,'.wav'))
    filename = filename[:9]+'-RAT-'+'D8F0-'+filename[-5:]
    shutil.copy('%s/%s%s' % (in_folder_RAT,filename,'.wav'), '%s/%s%s' % (out_folder_RAT,filename,'.wav'))

In [ ]:
# select one effect to analyse - change fx label string accordingly
df_808 = test_df[test_df['filename'].str.contains('-808-')]
df_ts9 = test_df[test_df['filename'].str.contains('-TS9-')]

In [ ]:
# select wrong predictions - change effect label number accordingly (e.g. 10 = SD1)
wrong_df_808 = df_808[df_808['fx_pred'].str.match('11')]
wrong_df_ts9 = df_ts9[df_ts9['fx_pred'].str.match('0')]

In [ ]:
wrong_df_808.shape

In [ ]:
wrong_df_808_mod.shape

In [ ]:
wrong_df_808_mod = wrong_df_808
wrong_df_808_mod['filename'] = wrong_df_808_mod['filename'].apply(lambda x: '%s%s%s' % (x[:14],'G',x[15:]))

In [ ]:
wrong_df_808_mod.head()